# COCO BBox/Mask -> CNN -> ANFIS 

Minimal COCO bbox/mask filtering -> CNN -> ANFIS pipeline.

Alur:
1. COCO segmentation dibuat menjadi label grid kepadatan.
2. Mask COCO dipakai sebagai region proposal/filter area kandidat sampah.
3. Grid background murni langsung dianggap Aman.
4. Grid kandidat masuk CNN pretrained untuk ekstraksi fitur visual.
5. PCA + MinMaxScaler menyederhanakan fitur CNN.
6. ANFIS menjadi classifier final: Aman / Tersebar / Kritis.

Catatan: pada data tanpa anotasi COCO, tahap mask filtering ini bisa diganti YOLO atau model segmentasi otomatis.


## Import Library


In [ ]:
from pathlib import Path
import random

import cv2
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from pycocotools.coco import COCO
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, Dataset
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0


## Konfigurasi utama


In [ ]:
from pathlib import Path

SEED = 42

IMAGE_DIR        = Path("/kaggle/input/datasets/dzikribassyril/dronewate-data/data/raw/images")
ANNOTATION_JSON  = Path("/kaggle/input/datasets/dzikribassyril/dronewate-data/data/raw/annotations/dronewaste_v2.0.json")
OUTPUT_DIR       = Path("/kaggle/working/models/coco_mask_cnn_anfis")  # ← wajib /kaggle/working

GRID_SIZE = 128
SAFE_DENSITY_MAX = 0.05
CRITICAL_DENSITY_MIN = 0.40
COCO_CANDIDATE_DENSITY_MIN = 0.0  # >0 berarti grid punya piksel mask sampah dari COCO

CNN_BATCH_SIZE = 128
ANFIS_BATCH_SIZE = 512
NUM_WORKERS = 0

PCA_COMPONENTS = 24
ANFIS_RULES = 12
ANFIS_EPOCHS = 120
ANFIS_LR = 0.003
ANFIS_WEIGHT_DECAY = 1e-5

SAFE_SAMPLE_RATIO = 1.0  # Jumlah grid Aman murni yang disampling relatif terhadap grid kandidat mask

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Label grid dari COCO


In [ ]:
def density_to_class(density):
    if density < SAFE_DENSITY_MAX:
        return 0
    if density < CRITICAL_DENSITY_MIN:
        return 1
    return 2


def build_grid_dataframe():
    coco = COCO(str(ANNOTATION_JSON))
    rows = []

    for image_id in coco.getImgIds():
        info = coco.loadImgs([image_id])[0]
        width = int(info["width"])
        height = int(info["height"])
        image_file = info["file_name"]

        mask = np.zeros((height, width), dtype=np.uint8)
        ann_ids = coco.getAnnIds(imgIds=[image_id])
        anns = coco.loadAnns(ann_ids)
        for ann in anns:
            mask = np.maximum(mask, coco.annToMask(ann).astype(np.uint8))

        for y in range(0, height - GRID_SIZE + 1, GRID_SIZE):
            for x in range(0, width - GRID_SIZE + 1, GRID_SIZE):
                grid_mask = mask[y:y + GRID_SIZE, x:x + GRID_SIZE]
                density = float(grid_mask.mean())
                rows.append({
                    "image_id": image_id,
                    "image_file": image_file,
                    "x": x,
                    "y": y,
                    "density": density,
                    "Y_Target": density_to_class(density),
                    "Grid_ID": f"{image_id}_{x}_{y}",
                })

    return pd.DataFrame(rows)


def split_by_image(df):
    image_ids = df["image_id"].drop_duplicates().values
    train_ids, temp_ids = train_test_split(image_ids, test_size=0.35, random_state=SEED)
    val_ids, test_ids = train_test_split(temp_ids, test_size=0.50, random_state=SEED)

    train_df = df[df["image_id"].isin(train_ids)].reset_index(drop=True)
    val_df = df[df["image_id"].isin(val_ids)].reset_index(drop=True)
    test_df = df[df["image_id"].isin(test_ids)].reset_index(drop=True)
    return train_df, val_df, test_df


## COCO bbox/mask filtering


In [ ]:
def add_coco_mask_filter(df):
    out = df.copy()
    out["mask_keep"] = out["density"] > COCO_CANDIDATE_DENSITY_MIN
    return out


def make_training_candidates(train_df):
    train_df = add_coco_mask_filter(train_df)

    mask_df = train_df[train_df["mask_keep"]]
    pure_safe_df = train_df[~train_df["mask_keep"]]

    n_safe = min(len(pure_safe_df), int(len(mask_df) * SAFE_SAMPLE_RATIO))
    sampled_safe = pure_safe_df.sample(n=n_safe, random_state=SEED)

    candidates = pd.concat([mask_df, sampled_safe], ignore_index=True)
    return candidates.sample(frac=1, random_state=SEED).reset_index(drop=True)


## CNN feature extractor


In [ ]:
class GridImageDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(str(IMAGE_DIR / row["image_file"]))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        grid = image[row["y"]:row["y"] + GRID_SIZE, row["x"]:row["x"] + GRID_SIZE]
        grid = Image.fromarray(grid)
        return self.transform(grid), int(row["Y_Target"])


class EfficientNetEmbedding(nn.Module):
    def __init__(self):
        super().__init__()
        base = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        self.features = base.features
        self.avgpool = base.avgpool

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        return torch.flatten(x, 1)


def extract_cnn_features(df, cnn_model, transform):
    dataset = GridImageDataset(df, transform)
    loader = DataLoader(
        dataset,
        batch_size=CNN_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda",
    )

    features = []
    labels = []
    cnn_model.eval()

    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(DEVICE)
            feats = cnn_model(batch_x).cpu().numpy()
            features.append(feats)
            labels.extend(batch_y.numpy().tolist())

    return np.vstack(features).astype(np.float32), np.array(labels, dtype=np.int64)


## ANFIS


In [ ]:
class ANFIS(nn.Module):
    def __init__(self, num_inputs, num_rules, num_classes):
        super().__init__()
        self.num_inputs = num_inputs
        self.num_rules = num_rules
        self.num_classes = num_classes

        self.mu = nn.Parameter(torch.randn(num_rules, num_inputs))
        self.sigma_raw = nn.Parameter(torch.ones(num_rules, num_inputs) * 0.5)
        self.consequent_weights = nn.Parameter(torch.randn(num_rules, num_inputs) * 0.05)
        self.consequent_bias = nn.Parameter(torch.zeros(num_rules))
        self.classifier = nn.Linear(num_rules, num_classes)

    def init_from_kmeans(self, X_train):
        kmeans = KMeans(n_clusters=self.num_rules, random_state=SEED, n_init=10)
        kmeans.fit(X_train)

        centers = torch.tensor(kmeans.cluster_centers_, dtype=torch.float32, device=self.mu.device)
        self.mu.data = centers

        sigma = torch.ones(self.num_rules, self.num_inputs, device=self.mu.device) * 0.25
        for i in range(self.num_rules):
            points = X_train[kmeans.labels_ == i]
            if len(points) > 1:
                sigma[i] = torch.tensor(points.std(axis=0) + 1e-3, dtype=torch.float32, device=self.mu.device)
        self.sigma_raw.data = torch.log(torch.exp(sigma) - 1.0)

    def forward(self, x):
        sigma = torch.nn.functional.softplus(self.sigma_raw) + 1e-6
        x_expanded = x.unsqueeze(1).expand(-1, self.num_rules, -1)
        membership = torch.exp(-0.5 * ((x_expanded - self.mu) / sigma) ** 2)

        firing = torch.prod(membership, dim=2)
        firing_norm = firing / (firing.sum(dim=1, keepdim=True) + 1e-8)

        consequent = (x_expanded * self.consequent_weights.unsqueeze(0)).sum(dim=2)
        consequent = consequent + self.consequent_bias.unsqueeze(0)
        rule_outputs = firing_norm * consequent

        logits = self.classifier(rule_outputs)
        return logits, firing_norm


def train_anfis(X_train, y_train):
    X_tensor = torch.tensor(X_train, dtype=torch.float32, device=DEVICE)
    y_tensor = torch.tensor(y_train, dtype=torch.long, device=DEVICE)

    model = ANFIS(X_train.shape[1], ANFIS_RULES, 3).to(DEVICE)
    model.init_from_kmeans(X_train)

    counts = np.bincount(y_train, minlength=3)
    weights = torch.tensor(len(y_train) / (3 * np.maximum(counts, 1)), dtype=torch.float32, device=DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = optim.Adam(model.parameters(), lr=ANFIS_LR, weight_decay=ANFIS_WEIGHT_DECAY)

    dataset = torch.utils.data.TensorDataset(X_tensor, y_tensor)
    loader = DataLoader(dataset, batch_size=ANFIS_BATCH_SIZE, shuffle=True)

    for epoch in range(ANFIS_EPOCHS):
        model.train()
        total_loss = 0.0

        for batch_x, batch_y in loader:
            logits, _ = model(batch_x)
            loss = criterion(logits, batch_y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * batch_x.size(0)

        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch + 1:03d} | loss {total_loss / len(dataset):.4f}")

    return model


def predict_anfis(model, X):
    X_tensor = torch.tensor(X, dtype=torch.float32, device=DEVICE)
    preds = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(X_tensor), ANFIS_BATCH_SIZE):
            logits, _ = model(X_tensor[start:start + ANFIS_BATCH_SIZE])
            preds.append(torch.argmax(logits, dim=1).cpu().numpy())
    return np.concatenate(preds)


def predict_full_pipeline(df, cnn_model, transform, pca, scaler, anfis_model):
    y_pred = np.zeros(len(df), dtype=np.int64)
    candidate_df = df[df["mask_keep"]].reset_index()

    if len(candidate_df) > 0:
        X_emb, _ = extract_cnn_features(candidate_df, cnn_model, transform)
        X_pca = pca.transform(X_emb)
        X_scaled = scaler.transform(X_pca).astype(np.float32)
        y_pred[candidate_df["index"].values] = predict_anfis(anfis_model, X_scaled)

    return y_pred


## Main


In [ ]:
print("Device:", DEVICE)
print("Membuat grid label dari COCO...")
df_grid = build_grid_dataframe()
train_df, val_df, test_df = split_by_image(df_grid)
val_df = add_coco_mask_filter(val_df)
test_df = add_coco_mask_filter(test_df)

print("Grid total:", len(df_grid))
print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))
print("Distribusi train:")
print(train_df["Y_Target"].value_counts().sort_index())
print("Grid kandidat mask val/test:", int(val_df["mask_keep"].sum()), int(test_df["mask_keep"].sum()))

train_candidates = make_training_candidates(train_df)
print("Grid kandidat training:", len(train_candidates))
print(train_candidates["Y_Target"].value_counts().sort_index())

print("Memuat CNN pretrained sebagai feature extractor...")
cnn_weights = EfficientNet_B0_Weights.DEFAULT
cnn_transform = cnn_weights.transforms()
cnn_model = EfficientNetEmbedding().to(DEVICE)

print("Ekstraksi fitur CNN untuk training ANFIS...")
X_train_emb, y_train = extract_cnn_features(train_candidates, cnn_model, cnn_transform)

pca_components = min(PCA_COMPONENTS, X_train_emb.shape[0], X_train_emb.shape[1])
pca = PCA(n_components=pca_components, random_state=SEED)
scaler = MinMaxScaler()

X_train_pca = pca.fit_transform(X_train_emb)
X_train = scaler.fit_transform(X_train_pca).astype(np.float32)

print("Training ANFIS...")
anfis_model = train_anfis(X_train, y_train)

print("Evaluasi full pipeline dengan COCO bbox/mask filtering...")
val_pred = predict_full_pipeline(val_df, cnn_model, cnn_transform, pca, scaler, anfis_model)
test_pred = predict_full_pipeline(test_df, cnn_model, cnn_transform, pca, scaler, anfis_model)

print("Val Accuracy:", accuracy_score(val_df["Y_Target"], val_pred))
print("Val Macro F1:", f1_score(val_df["Y_Target"], val_pred, average="macro"))
print(classification_report(val_df["Y_Target"], val_pred, target_names=["Aman", "Tersebar", "Kritis"]))

print("Test Accuracy:", accuracy_score(test_df["Y_Target"], test_pred))
print("Test Macro F1:", f1_score(test_df["Y_Target"], test_pred, average="macro"))
print(classification_report(test_df["Y_Target"], test_pred, target_names=["Aman", "Tersebar", "Kritis"]))
print("Confusion matrix test:")
print(confusion_matrix(test_df["Y_Target"], test_pred))

torch.save(anfis_model.state_dict(), OUTPUT_DIR / "anfis_coco_mask_cnn_minimal.pth")
joblib.dump(pca, OUTPUT_DIR / "pca_coco_mask_cnn_minimal.pkl")
joblib.dump(scaler, OUTPUT_DIR / "scaler_coco_mask_cnn_minimal.pkl")

train_candidates.to_csv(OUTPUT_DIR / "train_candidates.csv", index=False)
val_df.assign(pred=val_pred).to_csv(OUTPUT_DIR / "val_predictions.csv", index=False)
test_df.assign(pred=test_pred).to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)

print("Artefak disimpan di:", OUTPUT_DIR)
